# Notebook IV.2 — Residual armónico y Newton–Raphson

Este notebook acompaña la **Ventana computacional IV.2** del Capítulo IV.

Aquí pasamos de la integración temporal a la **formulación algebraica** del balance armónico.  
El objetivo es construir explícitamente un **residual armónico** para el oscilador de Duffing forzado y resolverlo mediante **Newton–Raphson**.

La idea central es mostrar que el balance armónico no es una receta aislada, sino una manera estructurada de imponer compatibilidad dinámica dentro de un subespacio espectral finito.

## Ejecutar este notebook en Google Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricomelgozajjesus/monografia-armonica/blob/main/python-lab/notebooks/Notebook_IV_02_Newton_balance.ipynb)

## 1. Modelo y aproximación de primer armónico

Trabajaremos con la ecuación de Duffing forzado:

$$
\ddot{x} + 2\zeta \omega_0 \dot{x} + \omega_0^2 x + \alpha x^3 = F\cos(\Omega t).
$$

Supondremos una aproximación de primer armónico de la forma

$$
x(t) \approx a\cos(\Omega t) + b\sin(\Omega t).
$$

Sustituyendo esta expresión en la ecuación diferencial y proyectando el residual sobre
$\cos(\Omega t)$ y $\sin(\Omega t)$ obtenemos un sistema algebraico no lineal para los coeficientes $(a,b)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Parámetros

Puedes modificar estos parámetros para explorar distintos regímenes de operación.

In [ ]:
zeta = 0.05
omega0 = 1.0
alpha = 1.0
F = 0.20
Omega = 1.0

## 3. Residual armónico de primer armónico

Si definimos

$$
x(t)=a\cos(\Omega t)+b\sin(\Omega t),
$$

la componente cúbica $x^3(t)$ introduce términos en la frecuencia fundamental y en el tercer armónico.
Si retenemos solamente la contribución proyectada sobre la frecuencia fundamental, el residual proyectado toma la forma

$$
R_1(a,b)=
\left(\omega_0^2-\Omega^2\right)a + 2\zeta\omega_0\Omega\, b + \frac{3}{4}\alpha(a^2+b^2)a - F,
$$

$$
R_2(a,b)=
\left(\omega_0^2-\Omega^2\right)b - 2\zeta\omega_0\Omega\, a + \frac{3}{4}\alpha(a^2+b^2)b.
$$

Resolver $R(a,b)=0$ equivale a encontrar una órbita periódica compatible dentro del subespacio armónico elegido.

In [ ]:
def residual(c, zeta, omega0, alpha, F, Omega):
    a, b = c
    rho2 = a*a + b*b
    r1 = (omega0**2 - Omega**2)*a + 2*zeta*omega0*Omega*b + 0.75*alpha*rho2*a - F
    r2 = (omega0**2 - Omega**2)*b - 2*zeta*omega0*Omega*a + 0.75*alpha*rho2*b
    return np.array([r1, r2], dtype=float)

def jacobian(c, zeta, omega0, alpha, F, Omega):
    a, b = c
    common = 0.75 * alpha
    j11 = (omega0**2 - Omega**2) + common * (3*a*a + b*b)
    j12 = 2*zeta*omega0*Omega + common * (2*a*b)
    j21 = -2*zeta*omega0*Omega + common * (2*a*b)
    j22 = (omega0**2 - Omega**2) + common * (a*a + 3*b*b)
    return np.array([[j11, j12],
                     [j21, j22]], dtype=float)

## 4. Implementar Newton–Raphson

Newton actualiza la incógnita $c=[a,b]^T$ mediante

$$
c_{k+1}=c_k - J(c_k)^{-1}R(c_k),
$$

donde $J(c)$ es el jacobiano del residual.

Guardaremos el historial para observar la convergencia.

In [ ]:
def newton_solve(c0, zeta, omega0, alpha, F, Omega, tol=1e-10, max_iter=30):
    c = np.array(c0, dtype=float)
    history = []

    for k in range(max_iter):
        R = residual(c, zeta, omega0, alpha, F, Omega)
        J = jacobian(c, zeta, omega0, alpha, F, Omega)
        norm_R = np.linalg.norm(R)
        history.append({
            "iter": k,
            "a": c[0],
            "b": c[1],
            "norm_R": norm_R
        })
        if norm_R < tol:
            return c, history, True
        delta = np.linalg.solve(J, -R)
        c = c + delta

    R = residual(c, zeta, omega0, alpha, F, Omega)
    history.append({
        "iter": max_iter,
        "a": c[0],
        "b": c[1],
        "norm_R": np.linalg.norm(R)
    })
    return c, history, False

## 5. Elegir condición inicial y resolver

Una de las lecciones importantes de esta ventana es que el método depende de la **semilla inicial**.  
En algunos regímenes, distintas semillas pueden conducir a ramas distintas o a problemas de convergencia.

In [ ]:
c0 = [0.4, 0.0]

sol, history, converged = newton_solve(c0, zeta, omega0, alpha, F, Omega)

print("Convergió:", converged)
print("Solución encontrada [a, b] =", sol)
print("Norma del residual final =", history[-1]["norm_R"])

In [ ]:
print("iter |        a |        b |   ||R||")
print("-" * 42)
for row in history:
    print(f"{row['iter']:4d} | {row['a']:8.5f} | {row['b']:8.5f} | {row['norm_R']:8.3e}")

## 6. Norma del residual por iteración

La caída de la norma del residual permite visualizar la convergencia de Newton–Raphson.

In [ ]:
iters = [row["iter"] for row in history]
norms = [row["norm_R"] for row in history]

plt.figure(figsize=(7.5, 4.5))
plt.semilogy(iters, norms, marker="o")
plt.xlabel("Iteración")
plt.ylabel(r"$\|R\|$")
plt.title("Convergencia de Newton–Raphson")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Reconstrucción temporal a partir de la solución

Una vez obtenidos los coeficientes $(a,b)$, podemos reconstruir la aproximación de primer armónico:

$$
x_1(t)=a\cos(\Omega t)+b\sin(\Omega t).
$$

Esto nos recuerda que la solución algebraica en realidad representa una órbita periódica aproximada.

In [ ]:
a, b = sol
T = 2*np.pi / Omega
t_plot = np.linspace(0, 4*T, 1200)
x1 = a*np.cos(Omega*t_plot) + b*np.sin(Omega*t_plot)

plt.figure(figsize=(10, 4.5))
plt.plot(t_plot, x1)
plt.xlabel("t")
plt.ylabel("x₁(t)")
plt.title("Reconstrucción temporal de la solución de primer armónico")
plt.grid(True, alpha=0.3)
plt.show()

A1 = np.sqrt(a*a + b*b)
phi1 = np.arctan2(-b, a)
print("Amplitud fundamental A1 =", A1)
print("Fase phi1 =", phi1, "rad")

## 8. Campo del residual en el plano $(a,b)$

También es útil mirar la geometría del residual en el plano de coeficientes.  
La solución de balance armónico aparece como un punto donde el residual se anula.

In [ ]:
a_vals = np.linspace(-1.2, 1.2, 31)
b_vals = np.linspace(-1.2, 1.2, 31)
A, B = np.meshgrid(a_vals, b_vals)

R1 = np.zeros_like(A)
R2 = np.zeros_like(B)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        rr = residual([A[i, j], B[i, j]], zeta, omega0, alpha, F, Omega)
        R1[i, j] = rr[0]
        R2[i, j] = rr[1]

plt.figure(figsize=(7, 7))
plt.quiver(A, B, -R1, -R2, angles="xy")
plt.plot(sol[0], sol[1], "ro", label="Solución Newton")
plt.xlabel("a")
plt.ylabel("b")
plt.title("Campo inducido por el residual proyectado")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Comentario conceptual

Esta formulación muestra con claridad una idea central del capítulo:

- el balance armónico convierte el problema dinámico en un problema algebraico no lineal;
- el residual proyectado mide la incompatibilidad de la órbita propuesta con la dinámica real;
- Newton–Raphson actúa como un mecanismo de corrección sobre los coeficientes armónicos.

En otras palabras, la órbita periódica se busca no integrando directamente en el tiempo, sino ajustando una arquitectura espectral hasta volverla dinámicamente admisible.

## 10. Exploraciones sugeridas

1. Cambia la semilla inicial `c0` y observa cómo cambia la convergencia.
2. Modifica `F` y estudia cómo cambia la amplitud fundamental.
3. Cambia `Omega` para explorar cercanía a resonancia.
4. Cambia el signo de `alpha` y compara hardening vs softening.
5. Intenta detectar casos en los que Newton converge lentamente o no converge.

## 11. Hacia el siguiente notebook

El siguiente paso natural es ir más allá del primer armónico y comparar aproximaciones con **uno, tres y cinco armónicos**.  
Allí la no linealidad aparecerá de forma aún más clara como un mecanismo de **acoplamiento espectral**.